# Functional Equivalence Heuristic Comparison

## Setup

In [1]:
# Install NL2CMD accuracy metric

! rm -rf bashlint/ metric/
! git clone https://github.com/IBM/clai.git
! git -C ./clai checkout nlc2cmd
! cp -r clai/utils/bashlint bashlint
! cp -r clai/utils/metric metric
! rm -rf clai
! sed -i 's/import collections/import collections.abc as collections/' bashlint/butils.py

Cloning into 'clai'...
remote: Enumerating objects: 4162, done.
remote: Counting objects: 100% (262/262), done.
remote: Compressing objects: 100% (206/206), done.
remote: Total 4162 (delta 69), reused 207 (delta 46), pack-reused 3900 (from 1)
Receiving objects: 100% (4162/4162), 131.85 MiB | 14.84 MiB/s, done.
Resolving deltas: 100% (2538/2538), done.
branch 'nlc2cmd' set up to track 'origin/nlc2cmd'.
Switched to a new branch 'nlc2cmd'
sed: 1: "bashlint/butils.py": undefined label 'ashlint/butils.py'


In [2]:
# Make results folder

! mkdir feh_results

mkdir: feh_results: File exists


In [3]:
import subprocess
import sys
import os

In [4]:
sys.path.insert(0, os.getcwd())
import collections
import collections.abc
for name in ['MutableSet', 'Mapping', 'Iterable', 'MutableMapping', 'Sequence']:
    if not hasattr(collections, name):
        setattr(collections, name, getattr(collections.abc, name))

In [5]:
# Imports

import os
import csv
import requests
import json

import warnings
warnings.filterwarnings('ignore', category=SyntaxWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from metric import metric_utils
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from sklearn.feature_extraction.text import TfidfVectorizer

from datasets import load_dataset
from icalfa import submit_command
from tabulate import tabulate
from tqdm import tqdm

/Users/dhwanichande/Desktop/NLP/Researchh Study/nl2sh_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setting bashlex grammar using file: /Users/dhwanichande/Desktop/NLP/Researchh Study/NL2SH/code/bashlint/grammar/grammar100.txt
Bashlint grammar set up (148 utilities)



In [17]:
# Ollama and OpenAI functions

system_prompt = """You are evaluating bash command equivalence. Answer ONLY with 'true' or 'false'.

Rules:
- If both commands accomplish the same task → answer: true
- If they accomplish different tasks → answer: false
- Only output one word: true or false
- Do NOT explain, write code, or add extra text

Examples:
Task: list files, Command 1: ls, Command 2: ls -l
true

Task: list files, Command 1: ls, Command 2: pwd
false"""

def ollama(prompt, model):
    url = "http://localhost:11434/api/chat"
    payload = json.dumps({
        "model": model,
        "messages": [
            {'role': 'system', 'content': system_prompt},
            {"role": "user", "content": prompt}
        ],
        "stream": False,
        "temperature": 0,
        "seed": 123,
        "options": {
            "num_predict": 10,      # Allow up to 10 tokens
            "num_ctx": 512,         # Smaller context
            "num_gpu": 1            # Use GPU
        }
    })
    
    try:
        response = requests.post(url, data=payload, timeout=60)
        
        if response.status_code != 200:
            print(f"❌ Error: {response.text}")
            return 0
        
        response_json = response.json()
        response_content = response_json['message']['content'].strip().lower()
        
        # Debug: Print first few responses to see what Llama is saying
        # Comment out after debugging
        # print(f"Llama response: '{response_content[:50]}'")
        
        # Take first word only
        first_word = response_content.split()[0] if response_content else ""
        
        # More flexible parsing
        if first_word in ['true', 'yes', '1']:
            return 1
        elif first_word in ['false', 'no', '0']:
            return 0
        else:
            # If unclear, check if "true" appears anywhere
            if 'true' in response_content and 'false' not in response_content:
                return 1
            else:
                return 0
                
    except Exception as e:
        print(f"❌ Exception: {e}")
        return 0

## Heuristics

In [18]:
# Bleu FEH
def bleu(index, prompt, ground_truth_command, model_command):
    gt_list = list(ground_truth_command)
    model_list = list(model_command)
    bleu_score = sentence_bleu([gt_list], model_list, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=SmoothingFunction().method1)
    return bleu_score > 0.75

# NL2CMD FEH
def nl2cmd(index, prompt, ground_truth_command, model_command):
    nl2cmd_score = metric_utils.compute_metric(predicted_cmd=model_command, predicted_confidence=1, ground_truth_cmd=ground_truth_command)
    # scale score to [0-1]
    nl2cmd_score = (nl2cmd_score + 1)/2
    return nl2cmd_score > 0.75

# TF-IDF FEH
def tfidf(index, prompt, ground_truth_command, model_command):
    try:
        vect = TfidfVectorizer()
        tfidf = vect.fit_transform([ground_truth_command, model_command])
        similarity = tfidf * tfidf.T
        tfidf_score = similarity.toarray()[0][1]
    except:
        tfidf_score = 0
    return tfidf_score > 0.75

# Execution + TF-IDF FEH
def exec_tfidf(index, prompt, ground_truth_command, model_command):
    score = submit_command(index=index, command=model_command, eval_mode="tfidf")
    return score == 1

# mxbai-embed FEH
embedding_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")
def mxbai_embed(index, prompt, ground_truth_command, model_command):
    embeddings = embedding_model.encode([ground_truth_command, model_command])
    cosine_similarity = cos_sim(embeddings[0], embeddings[1])
    embedding_score = cosine_similarity.item()
    return embedding_score > 0.75

# Execution + mxbai-embed FEH
def exec_mxbai_embed(index, prompt, ground_truth_command, model_command):
    score = submit_command(index=index, command=model_command, eval_mode="embed", eval_param=0.75)
    return score == 1

# Llama3.2-1b-Instruct FEH
# Llama3.2-1b-Instruct FEH
def llama3(index, prompt, ground_truth_command, model_command):
    score = ollama(f"Task: {prompt}, Ground Truth Command: {ground_truth_command}, Model Command: {model_command}", "llama3.2:1b") 
    return score == 1

# Execution + Llama3.1-1b-Instruct FEH
def exec_llama3(index, prompt, ground_truth_command, model_command):
    score = submit_command(index=index, command=model_command, eval_mode="ollama", eval_param="llama3.2:1b") 
    return score == 1

# GPT-3.5-Turbo FEH
#def gpt3(index, prompt, ground_truth_command, model_command):
#    score = openai(f"Task: {prompt}, Ground Truth Command: {ground_truth_command}, Model Command: {model_command}", "gpt-3.5-turbo-0125")
#    return score == 1

# Execution + GPT-3.5-Turbo FEH
#def exec_gpt3(index, prompt, ground_truth_command, model_command):
#    score = submit_command(index=index, command=model_command, eval_mode="openai", eval_param="gpt-3.5-turbo-0125")
#    return score == 1

# GPT-4 FEH
#def gpt4(index, prompt, ground_truth_command, model_command):
#    score = openai(f"Task: {prompt}, Ground Truth Command: {ground_truth_command}, Model Command: {model_command}", "gpt-4-0613")
#    return score == 1

# Execution + GPT-4 FEH
#def exec_gpt4(index, prompt, ground_truth_command, model_command):
#    score = submit_command(index=index, command=model_command, eval_mode="openai", eval_param="gpt-4-0613")
#    return score == 1

## Comparison

In [19]:
file_names = ['bleu.csv', 'nl2cmd.csv', 'tfidf.csv', 'exec_tfidf.csv', 'mxbai_embed.csv', 'exec_mxbai_embed.csv', 'llama3.csv', 'exec_llama3.csv']

fehs = [bleu, nl2cmd, tfidf, exec_tfidf, mxbai_embed, exec_mxbai_embed, llama3, exec_llama3]

dataset = load_dataset("westenfelder/NL2SH-ALFA", "test", split="train")

# rotate dataset by 10 positions to create non-equivalent commands
indices = list(range(300))
shuffled = indices[-10:] + indices[:-10]

for file_index, name in enumerate(file_names):
    if os.path.exists(f"feh_results/{name}"):
        print(f"{name} already exists, skipping")
        continue

    print(f"Creating {name}")

    with open(f"feh_results/{name}", mode='w') as file:
        writer = csv.writer(file)
        writer.writerow(['Task', 'Ground Truth Command', 'Model Command', 'Functional Equivalence', 'Heuristic Output'])

        for data_index, row in tqdm(enumerate(dataset), total=len(dataset)):
            prompt = row['nl']
            ground_truth_command = row['bash']
            func_equiv_command = row['bash2']
            not_func_equiv_command = dataset[shuffled[data_index]]["bash"]
            # get heuristic output
            func_equiv_feh_output = fehs[file_index](data_index, prompt, ground_truth_command, func_equiv_command)
            not_func_equiv_feh_output = fehs[file_index](data_index, prompt, ground_truth_command, not_func_equiv_command)

            writer.writerow([prompt, ground_truth_command, func_equiv_command, "True", func_equiv_feh_output])
            writer.writerow([prompt, ground_truth_command, not_func_equiv_command, "False", not_func_equiv_feh_output])

bleu.csv already exists, skipping
nl2cmd.csv already exists, skipping
tfidf.csv already exists, skipping
exec_tfidf.csv already exists, skipping
mxbai_embed.csv already exists, skipping
exec_mxbai_embed.csv already exists, skipping
Creating llama3.csv


100%|██████████| 300/300 [28:45<00:00,  5.75s/it]


Creating exec_llama3.csv


100%|██████████| 300/300 [31:26<00:00,  6.29s/it] 


In [21]:
import csv
from tabulate import tabulate

results = [["Heuristic", "TP", "FP", "TN", "FN", "Precision", "Recall", "F1 Score", "Accuracy"]]

heuristics = ['bleu.csv', 'nl2cmd.csv', 'tfidf.csv', 'exec_tfidf.csv', 'mxbai_embed.csv', 'exec_mxbai_embed.csv', 'llama3.csv', 'exec_llama3.csv']

for filename in heuristics:
    filepath = f'feh_results/{filename}'
    
    try:
        with open(filepath, mode='r') as file:
            reader = csv.reader(file)
            next(reader)  # Skip header
            
            true_positive = false_positive = true_negative = false_negative = 0
            
            for row in reader:
                if row[3] == "True" and row[4] == "True":
                    true_positive += 1
                elif row[3] == "True" and row[4] == "False":
                    false_negative += 1
                elif row[3] == "False" and row[4] == "True":
                    false_positive += 1
                elif row[3] == "False" and row[4] == "False":
                    true_negative += 1
            
            # Safe division with zero checks
            precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
            recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
            accuracy = (true_positive + true_negative) / (true_positive + true_negative + false_positive + false_negative)
            
            results.append([
                filename.replace('.csv', ''),
                true_positive, false_positive, true_negative, false_negative,
                f"{precision:.2f}", f"{recall:.2f}", f"{f1:.2f}", f"{accuracy:.2f}"
            ])
    
    except FileNotFoundError:
        print(f"⚠️  {filename} not found, skipping")

print(tabulate(results, headers="firstrow", tablefmt="grid"))

+------------------+------+------+------+------+-------------+----------+------------+------------+
| Heuristic        |   TP |   FP |   TN |   FN |   Precision |   Recall |   F1 Score |   Accuracy |
+==================+======+======+======+======+=============+==========+============+============+
| bleu             |  116 |    1 |  299 |  184 |        0.99 |     0.39 |       0.56 |       0.69 |
+------------------+------+------+------+------+-------------+----------+------------+------------+
| nl2cmd           |   60 |    1 |  299 |  240 |        0.98 |     0.2  |       0.33 |       0.6  |
+------------------+------+------+------+------+-------------+----------+------------+------------+
| tfidf            |  137 |    1 |  299 |  163 |        0.99 |     0.46 |       0.63 |       0.73 |
+------------------+------+------+------+------+-------------+----------+------------+------------+
| exec_tfidf       |  195 |    2 |  298 |  105 |        0.99 |     0.65 |       0.78 |       0.82 |
